# Arbol de Decision para Clasificacion Binaria

## Objetivo
Implementar un modelo de Arbol de Decision para predecir la direccion del cambio
de tasas de cambio (sube = 1, baja/estable = 0).

In [ ]:
# Celda 1: Configuracion del entorno
import sys
import os

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv(os.path.join(PROJECT_ROOT, ".env.local"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Celda 2: Carga de datos
import psycopg2

def get_connection():
    db_host = os.getenv("DB_HOST", "postgres")
    
    if db_host == "postgres":
        host = "postgres"
        port = os.getenv("DB_PORT", "5432")
    else:
        host = "localhost"
        port = os.getenv("DB_PORT", "5433")
    return psycopg2.connect(
        host=host,
        port=port,
        dbname=os.getenv("DB_NAME", "exchange_db"),
        user=os.getenv("DB_USER", "postgres"),
        password=os.getenv("DB_PASSWORD", "postgres"),
        sslmode="disable"
    )

conn = get_connection()

query = """
SELECT 
    er.id,
    c1.code AS base_currency,
    c2.code AS target_currency,
    er.rate,
    er.timestamp
FROM exchange_rates er
JOIN currencies c1 ON er.base_currency_id = c1.id
JOIN currencies c2 ON er.target_currency_id = c2.id
ORDER BY er.timestamp;
"""

df = pd.read_sql(query, conn)
conn.close()
df["timestamp"] = pd.to_datetime(df["timestamp"])
df.head()

## Preparacion de Datos

In [ ]:
# Celda 3: Crear variable objetivo binaria
df_model = df.copy()
df_model = df_model.sort_values(['target_currency', 'timestamp'])

df_model['rate_lag1'] = df_model.groupby('target_currency')['rate'].shift(1)
df_model['rate_lag2'] = df_model.groupby('target_currency')['rate'].shift(2)
df_model['rate_lag3'] = df_model.groupby('target_currency')['rate'].shift(3)
df_model['rate_lag5'] = df_model.groupby('target_currency')['rate'].shift(5)

df_model['rate_change'] = df_model['rate'] - df_model['rate_lag1']
df_model['pct_change'] = df_model['rate_change'] / df_model['rate_lag1'] * 100

df_model['rolling_mean_3'] = df_model.groupby('target_currency')['rate'].transform(lambda x: x.rolling(3).mean())
df_model['rolling_std_3'] = df_model.groupby('target_currency')['rate'].transform(lambda x: x.rolling(3).std())
df_model['rolling_mean_5'] = df_model.groupby('target_currency')['rate'].transform(lambda x: x.rolling(5).mean())
df_model['rolling_mean_7'] = df_model.groupby('target_currency')['rate'].transform(lambda x: x.rolling(7).mean())

df_model['volatility_3'] = df_model['rolling_std_3'] / df_model['rolling_mean_3'] * 100
df_model['volatility_5'] = df_model.groupby('target_currency')['rate'].transform(lambda x: x.rolling(5).std()) / df_model['rolling_mean_5'] * 100

df_model['direction'] = (df_model['rate_change'] > 0).astype(int)

df_model = df_model.dropna(subset=['rate_lag1', 'pct_change', 'direction'])
df_model = df_model.fillna(0)
print(f"Forma del dataset: {df_model.shape}")
print(f"Distribucion de la variable objetivo:\n{df_model['direction'].value_counts()}")

## Entrenamiento del Modelo

In [ ]:
# Celda 4: Division de datos
features = ['rate_lag1', 'rate_lag2', 'rate_lag3', 'rate_lag5',
            'pct_change', 'rolling_mean_3', 'rolling_std_3', 'rolling_mean_5', 'rolling_mean_7',
            'volatility_3', 'volatility_5']

X = df_model[features]
y = df_model['direction']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Entrenamiento: {X_train.shape[0]} | Prueba: {X_test.shape[0]}")
print(f"Clase 1 en train: {y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"Clase 1 en test: {y_test.sum()} ({y_test.mean()*100:.1f}%)")

In [ ]:
# Celda 5: Entrenamiento del Arbol de Clasificacion
dt_classifier = DecisionTreeClassifier(
    max_depth=8,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42,
    class_weight='balanced'
)

dt_classifier.fit(X_train, y_train)

y_pred = dt_classifier.predict(X_test)
y_prob = dt_classifier.predict_proba(X_test)[:, 1]

print("Metricas del Modelo:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

## Visualizacion del Arbol

In [ ]:
# Celda 6: Visualizacion del Arbol
plt.figure(figsize=(20, 10))
plot_tree(dt_classifier, 
          feature_names=features,
          class_names=['Baja/Estable', 'Sube'],
          filled=True,
          rounded=True,
          fontsize=8)
plt.title('Arbol de Decision - Clasificacion Binaria')
plt.tight_layout()
plt.show()

## Matriz de Confusion

In [ ]:
# Celda 7: Matriz de Confusion
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap='Greens')
plt.title('Matriz de Confusion')
plt.colorbar()

classes = ['Baja/Estable (0)', 'Sube (1)']
tick_marks = np.arange(len(classes))
plt.xticks(tick_marks, classes)
plt.yticks(tick_marks, classes)

for i in range(2):
    for j in range(2):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=12)

plt.ylabel('Real')
plt.xlabel('Predicho')
plt.tight_layout()
plt.show()

print("\nReporte de Clasificacion:")
print(classification_report(y_test, y_pred, target_names=classes))

## Feature Importance

In [ ]:
# Celda 8: Feature Importance
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': dt_classifier.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'], color='green')
plt.xlabel('Importancia')
plt.ylabel('Caracteristica')
plt.title('Feature Importance - Arbol de Clasificacion')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

feature_importance

## Validacion Cruzada

In [ ]:
# Celda 9: Cross-Validation
cv_scores = cross_val_score(dt_classifier, X, y, cv=5, scoring='accuracy')

print("Validacion Cruzada (5 folds):")
print(f"Accuracy por fold: {cv_scores}")
print(f"Accuracy promedio: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## Optimizacion de Hiperparametros

In [ ]:
# Celda 10: GridSearchCV
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [4, 6, 8, 10, 12],
    'min_samples_split': [10, 20, 30],
    'min_samples_leaf': [5, 10, 15]
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Mejores parametros:", grid_search.best_params_)
print("Mejor F1 Score:", grid_search.best_score_)

In [ ]:
# Celda 11: Modelo optimizado
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)
y_prob_best = best_model.predict_proba(X_test)[:, 1]

print("Metricas del Modelo Optimizado:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_best):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_best):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_best):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred_best):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_best):.4f}")

## Conclusion

In [ ]:
# Celda 12: Resumen
try:
    y_pred_best_exists = 'y_pred_best' in globals() and y_pred_best is not None
except:
    y_pred_best_exists = False

if y_pred_best_exists:
    resultado = pd.DataFrame({
        'Modelo': ['Arbol Clasificacion Default', 'Arbol Clasificacion Optimizado'],
        'Accuracy': [accuracy_score(y_test, y_pred), accuracy_score(y_test, y_pred_best)],
        'Precision': [precision_score(y_test, y_pred), precision_score(y_test, y_pred_best)],
        'Recall': [recall_score(y_test, y_pred), recall_score(y_test, y_pred_best)],
        'F1': [f1_score(y_test, y_pred), f1_score(y_test, y_pred_best)],
        'AUC-ROC': [roc_auc_score(y_test, y_prob), roc_auc_score(y_test, y_prob_best)]
    })
else:
    resultado = pd.DataFrame({
        'Modelo': ['Arbol Clasificacion Default'],
        'Accuracy': [accuracy_score(y_test, y_pred)],
        'Precision': [precision_score(y_test, y_pred)],
        'Recall': [recall_score(y_test, y_pred)],
        'F1': [f1_score(y_test, y_pred)],
        'AUC-ROC': [roc_auc_score(y_test, y_prob)]
    })
    print("Nota: Modelo optimizado no disponible")

print("=" * 60)
print("RESUMEN DE MODELOS")
print("=" * 60)
resultado

In [ ]:
# Celda 13: Guardar modelo
import joblib

joblib.dump(best_model, 'arbol_clasificacion_model.pkl')
print("Modelo guardado: arbreclasificacionmodel.pkl")